# TerraFM LULC Segmentation (S1 + S2)

19-class Sentinel-2 + Sentinel-1 LULC segmentation using **TerraFM-Base + UPerNet**.
Run cells **in order**. Data is downloaded with `gdown` — no Drive mount needed.

### Quick-start order
Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 → 15

In [ ]:
# =====================================================================
# Cell 1 — Install dependencies
# =====================================================================
!pip install -q rasterio>=1.3.9 timm>=0.9.12 transformers>=4.40.0 \
    huggingface_hub>=0.22.0 gdown tqdm pandas matplotlib seaborn scikit-learn
print('✓ Dependencies installed.')

In [ ]:
# =====================================================================
# Cell 2 — Download dataset from Google Drive using gdown
#
# Replace GDRIVE_FILE_ID with your actual Google Drive file ID.
# The file should be a zip archive containing:
#   data/S1/, data/S2/, data/selected_reference_map/, data/file.csv
#
# Example: if your Drive URL is
#   https://drive.google.com/file/d/1AbCdEfGhIjKl/view
# then GDRIVE_FILE_ID = '1AbCdEfGhIjKl'
# =====================================================================
import os

GDRIVE_FILE_ID = 'YOUR_GDRIVE_FILE_ID_HERE'   # <-- CHANGE THIS
DOWNLOAD_PATH  = '/content/dataset.zip'
DATA_DIR       = '/content/data'

if not os.path.exists(DATA_DIR):
    import gdown
    gdown.download(id=GDRIVE_FILE_ID, output=DOWNLOAD_PATH, quiet=False)
    !unzip -q {DOWNLOAD_PATH} -d /content/
    print(f'✓ Dataset extracted to {DATA_DIR}')
else:
    print(f'✓ {DATA_DIR} already exists – skipping download.')

!ls {DATA_DIR}

In [ ]:
# =====================================================================
# Cell 3 — Configure paths and import CFG
# =====================================================================
import sys, os

# Path to project code (adjust if your code is in a subfolder)
PROJECT_DIR = '/content/segmentation'
if not os.path.exists(PROJECT_DIR):
    # Clone or copy project files here if not already present
    raise FileNotFoundError(
        f'{PROJECT_DIR} not found. '
        'Upload the project .py files to /content/segmentation/ first.'
    )
sys.path.insert(0, PROJECT_DIR)

from config import CFG
from utils import setup_logging, check_amp_support, save_config

setup_logging('INFO')

# ---- Data paths ----
CFG.data_root = '/content/data'
CFG.s2_dir    = '/content/data/S2'
CFG.s1_dir    = '/content/data/S1'
CFG.ref_dir   = '/content/data/selected_reference_map'
# Point to the CLEANED CSV (output of Cell 5 / clean_dataset.py)
CFG.csv_file  = '/content/data/file_clean.csv'

# ---- Output paths (all on Colab VM; save important outputs yourself) ----
CFG.output_dir     = '/content/outputs'
CFG.checkpoint_dir = '/content/outputs/checkpoints'
CFG.results_dir    = '/content/outputs/results'
CFG.split_dir      = '/content/outputs/splits'
CFG.weights_dir    = '/content/outputs/weights'
CFG.norm_stats_file    = '/content/outputs/norm_stats.json'
CFG.class_mapping_file = '/content/outputs/class_mapping.json'
CFG.final_model_path   = '/content/outputs/terrafm_lulc_model.pth'

# ---- AMP dtype (FP16 for T4) ----
amp = check_amp_support()
CFG.amp_dtype = amp if amp != 'none' else 'fp16'

CFG.ensure_dirs()
save_config(CFG, '/content/outputs/config.json')
print(f'✓ Config ready.  AMP: {CFG.amp_dtype}  |  Input channels: {CFG.total_in_channels}')

In [ ]:
# =====================================================================
# Cell 4 — Verify TerraFM weights are accessible on HuggingFace Hub
# =====================================================================
from huggingface_hub import list_repo_files
try:
    files = list(list_repo_files(CFG.terrafm_hub_id))
    print(f'✓ {CFG.terrafm_hub_id} is accessible ({len(files)} files)')
    for f in files[:10]:
        print(f'  {f}')
except Exception as e:
    print(f'⚠  Could not list HF repo: {e}')
    print('   Weights will be downloaded when training starts.')

In [ ]:
# =====================================================================
# Cell 5 — Clean the dataset  (run ONCE, first time only)
#
# Reads /content/data/file.csv
# Validates every S1/S2 folder and reference TIFF for corruption
# Deletes corrupt folders from disk
# Writes /content/data/file_clean.csv  and  /content/outputs/corrupted_patches.txt
# =====================================================================
import os

CLEAN_CSV = '/content/data/file_clean.csv'
if not os.path.exists(CLEAN_CSV):
    !python {PROJECT_DIR}/clean_dataset.py \
        --csv  /content/data/file.csv \
        --s1   /content/data/S1 \
        --s2   /content/data/S2 \
        --ref  /content/data/selected_reference_map \
        --out_csv  {CLEAN_CSV} \
        --log  /content/outputs/corrupted_patches.txt
    print(f'✓ Cleaning complete. Cleaned CSV: {CLEAN_CSV}')
else:
    print(f'✓ {CLEAN_CSV} already exists – skipping cleaning.')

import pandas as pd
df = pd.read_csv(CLEAN_CSV)
print(f'   Rows in file_clean.csv: {len(df)}')
print(df.head(3))

In [ ]:
# =====================================================================
# Cell 6 — Prepare dataset: splits + normalization + class stats
#          Run ONCE after cleaning. Takes ~5-10 min for 50k patches.
# =====================================================================
from prepare_dataset import prepare_all

prep = prepare_all(csv_file=CLEAN_CSV)

TRAIN_CSV    = prep['train_csv']
VAL_CSV      = prep['val_csv']
TEST_CSV     = prep['test_csv']
S2_MEAN      = prep['s2_mean']
S2_STD       = prep['s2_std']
S1_MEAN      = prep['s1_mean']
S1_STD       = prep['s1_std']
RAW_TO_TRAIN = prep['raw_to_train']
PIXEL_COUNTS = prep['pixel_counts']

print(f'\n✓ Preparation complete.')
print(f'  Train: {TRAIN_CSV}  Val: {VAL_CSV}  Test: {TEST_CSV}')

In [ ]:
# =====================================================================
# Cell 7 — Reload previously computed stats (use after session restart)
#          Run this instead of Cell 6 if preparation already finished.
# =====================================================================
import os, json
from utils import load_norm_stats, load_class_mapping, load_class_stats

TRAIN_CSV = '/content/outputs/splits/train.csv'
VAL_CSV   = '/content/outputs/splits/val.csv'
TEST_CSV  = '/content/outputs/splits/test.csv'

with open(CFG.norm_stats_file) as f:
    ns = json.load(f)
S2_MEAN = ns['s2_mean'];  S2_STD = ns['s2_std']
S1_MEAN = ns['s1_mean'];  S1_STD = ns['s1_std']

RAW_TO_TRAIN = load_class_mapping(CFG.class_mapping_file)

cs_path = '/content/outputs/class_stats.json'
cs      = load_class_stats(cs_path)
PIXEL_COUNTS = cs['pixel_counts']

print('✓ Stats reloaded.')
print(f'  S2 mean (first 3): {S2_MEAN[:3]}')
print(f'  S1 mean (dB):      {S1_MEAN}')

In [ ]:
# =====================================================================
# Cell 8 — Visualize training examples (S2 RGB + reference mask)
# =====================================================================
import pandas as pd
import matplotlib.pyplot as plt
from dataset import S1S2LULCDataset, load_records_from_csv
from visualize import plot_qualitative_sample

records = load_records_from_csv(TRAIN_CSV)[:4]
ds = S1S2LULCDataset(
    records=records, s2_root=CFG.s2_dir, s1_root=CFG.s1_dir,
    ref_root=CFG.ref_dir,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN, augment=False,
)
for i in range(min(2, len(ds))):
    fused, mask = ds[i]
    plot_qualitative_sample(
        s2=fused[:12], gt_mask=mask, pred_mask=mask,   # show GT as both
        class_names=CFG.class_names, class_colors=CFG.class_colors,
        ignore_index=CFG.ignore_index,
        save_path=f'/content/outputs/preview_{i}.png',
        norm_mean=S2_MEAN, norm_std=S2_STD,
    )
print('✓ Sample previews saved to /content/outputs/')

In [ ]:
# =====================================================================
# Cell 9 — Phase 0: Smoke test (MANDATORY before full training)
#
# Overfits 64 samples to verify the full S1+S2 pipeline.
# Expected: training loss drops below 1.0 within 30 epochs.
# If not: there is a bug in data loading, class mapping, or the model.
# =====================================================================
from train import run_smoke_test

run_smoke_test(
    train_csv=TRAIN_CSV, val_csv=VAL_CSV,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN,
)

In [ ]:
# =====================================================================
# Cell 10 — Phase 1: Pilot training (~2000 samples)
#
# Purpose: validate real training behavior, benchmark T4 speed/memory.
# Expected: val mIoU > 0.2 after 10 epochs (encoder frozen).
# =====================================================================
import pandas as pd, tempfile, os
from train import train as run_training

df = pd.read_csv(TRAIN_CSV).head(CFG.phase1_samples)
with tempfile.TemporaryDirectory() as tmp:
    p_csv = os.path.join(tmp, 'pilot_train.csv')
    df.to_csv(p_csv, index=False)
    run_training(
        train_csv=p_csv, val_csv=VAL_CSV,
        s2_mean=S2_MEAN, s2_std=S2_STD,
        s1_mean=S1_MEAN, s1_std=S1_STD,
        raw_to_train=RAW_TO_TRAIN, class_pixel_counts=PIXEL_COUNTS,
        epochs=CFG.phase1_epochs, batch_size=CFG.phase1_batch_size,
        freeze_stage=0, phase_name='phase1',
    )

In [ ]:
# =====================================================================
# Cell 11 — Phase 2: Full training
#
# Epochs 1-5:  Encoder frozen (Stage 0) — decoder warms up
# Epoch  6+:   Last 4 ViT blocks + patch embed unfrozen (Stage 1)
# Early stopping patience = 8 epochs
#
# To RESUME after a crash:
#   CFG.resume_from = '/content/outputs/checkpoints/last_checkpoint.pth'
# =====================================================================
from train import train as run_training

# CFG.resume_from = '/content/outputs/checkpoints/last_checkpoint.pth'

best_ckpt = run_training(
    train_csv=TRAIN_CSV, val_csv=VAL_CSV,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN, class_pixel_counts=PIXEL_COUNTS,
    epochs=CFG.epochs, batch_size=CFG.batch_size,
    freeze_stage=0, phase_name='phase2',
    resume_from=CFG.resume_from,
)
print(f'Best checkpoint: {best_ckpt}')

In [ ]:
# =====================================================================
# Cell 12 — Evaluate best checkpoint on test set
# =====================================================================
from evaluate import evaluate
import os

best_ckpt = os.path.join(CFG.checkpoint_dir, 'best_model.pth')
results = evaluate(
    model_path=best_ckpt, test_csv=TEST_CSV,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN,
    class_names=CFG.class_names,
)
print(f'Test mIoU: {results["mean_iou"]:.4f}')

In [ ]:
# =====================================================================
# Cell 13 — Plot training curves
# =====================================================================
from visualize import plot_training_curves
from IPython.display import Image

history_csv = '/content/outputs/training_history_phase2.csv'
curves_path = '/content/outputs/results/training_curves.png'
plot_training_curves(history_csv, curves_path)
Image(curves_path)

In [ ]:
# =====================================================================
# Cell 14 — Export final model file (terrafm_lulc_model.pth)
#
# This single file contains everything needed for inference:
# weights + architecture config + norm stats + class mapping.
# =====================================================================
import os, torch
from model import build_model, save_final_model
from checkpoint import load_checkpoint

best_ckpt = os.path.join(CFG.checkpoint_dir, 'best_model.pth')
model = build_model(freeze_stage=2)
load_checkpoint(best_ckpt, model)

save_final_model(
    model=model,
    path=CFG.final_model_path,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN,
    class_names=CFG.class_names,
    class_colors=CFG.class_colors,
)
print(f'✓ Final model: {CFG.final_model_path}')

# Download the final model to your machine:
from google.colab import files
files.download(CFG.final_model_path)

In [ ]:
# =====================================================================
# Cell 15 — Inference on a new S1+S2 patch pair
# =====================================================================
from inference import patch_inference
from IPython.display import Image

# Set these to point to your new S2 and S1 patch directories
NEW_S2_DIR = '/content/data/S2/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_37_88'
NEW_S1_DIR = '/content/data/S1/S1B_IW_GRDH_1SDV_20170612T165809_33UUP_37_88'

outputs = patch_inference(
    model_path=CFG.final_model_path,
    s2_patch_dir=NEW_S2_DIR,
    s1_patch_dir=NEW_S1_DIR,
    output_dir='/content/outputs/inference',
)
Image(outputs['vis_png'])